In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
from natsort import natsorted
import plotly.graph_objects as go
from os.path import join as pjoin
from numpy.random import MT19937, SeedSequence, RandomState

sys.path.append("../../")
import circletrack_behavior as ctb
import plotting_functions as pf

In [ ]:
## Settings
parent_dir = 'CircleTrack_Inhibition'
experiment_dir = 'Inhibition3'
lin_path = f'../../../{parent_dir}/{experiment_dir}/output/lin_behav/'
circle_path = f'../../../{parent_dir}/{experiment_dir}/output/behav/'
fig_path = f'../../../{parent_dir}/{experiment_dir}/intermediate_figures'
maze_info = pd.read_csv(f'../../../{parent_dir}/{experiment_dir}/maze_yml/maze_info.csv')
chance_color = '#7d7d7d'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
two_group_colors = ['darkorchid', 'midnightblue']
group_colors_dict = {'DREADD': 'darkorchid', 'Control': 'midnightblue'}
error_dict = {'DREADD': 'rgba(153,50,204,0.4)', 'Control': 'rgba(0,41,102,0.4)'}

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:
## Randomize port numbers for each context for each mouse
rs = RandomState(MT19937(SeedSequence(24601)))
context_list = ['A', 'B', 'C']
port_list = [0, 1, 2, 3]
mouse_list = [f'inh{x}' for x in np.arange(21, 37)]
potential_combinations = [
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward7'], ['reward4', 'reward8']],
    [['reward1', 'reward6'], ['reward2', 'reward5'], ['reward3', 'reward8'], ['reward4', 'reward7']],
    [['reward1', 'reward4'], ['reward2', 'reward7'], ['reward3', 'reward6'], ['reward5', 'reward8']],
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward8'], ['reward4', 'reward7']],
]

output = {}
for mouse in mouse_list:
    context_ports = {'A': [], 'B': [], 'C': [], 'D': []}
    randcont = rs.randint(0, len(context_list))
    randports = rs.choice(port_list, size=4, replace=False)
    for context, ports in zip(context_list, randports):
        context_ports[context].append(potential_combinations[randcont][ports])
    output[f'{mouse}'] = context_ports
port_df = pd.DataFrame(output)

### Circle track lick accuracy and rewards.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'sex': [], 'group': [], 'session': [], 'lick_accuracy': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mouse_path = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    if group == 'Control':
        group = 'mCherry'
    elif group == 'DREADD':
        group = 'hM3Dq'
    for idx, session in enumerate(natsorted(os.listdir(mouse_path))):
        behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
        behav = behav[~behav['probe']]
        reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
        pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
        circletrack_results['mouse'].append(mouse)
        circletrack_results['day'].append(idx+1)
        circletrack_results['sex'].append(sex)
        circletrack_results['group'].append(group)
        circletrack_results['session'].append(np.unique(behav['session'])[0])
        circletrack_results['lick_accuracy'].append(pc_thresh5)
        circletrack_results['rewards'].append(np.sum(behav['water']))
ct_df = pd.DataFrame(circletrack_results)

In [ ]:
## Plot lick accuracy across days up to first day in B
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 7], x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle', 'circle'],
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_B1.png'), width=500, height=500)

In [ ]:
## Plot rewards across days for both groups up to first day in B
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 7], x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle', 'circle'], plot_datapoints=True,
                                   x_title='Day', y_title='Rewards', titles=[''], height=500, width=500)
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_B1.png'), width=500, height=500)

In [ ]:
## Plot lick accuracy across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle', 'circle'],
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot rewards across days for both groups
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle', 'circle'], plot_datapoints=True,
                                   x_title='Day', y_title='Rewards', titles=[''], height=500, width=500)
fig.show()

### Look at probe accuracy.

In [ ]:
lick_dict_probe = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'session': [], 
                   'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(natsorted(os.listdir(mpath))):
        behav = pd.read_feather(pjoin(mpath, session))
        if any(behav['probe']):
            behav_probe = behav[behav['probe']]
            behav_no_probe = behav[~behav['probe']]
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
            percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            lick_dict_probe['mouse'].append(mouse)
            lick_dict_probe['experiment'].append(behav['cohort'].unique()[0])
            lick_dict_probe['sex'].append(sex)
            lick_dict_probe['group'].append(group)
            lick_dict_probe['day'].append(idx+1)
            lick_dict_probe['session'].append(np.unique(behav['session_two'])[0])
            lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
            lick_dict_probe['probe_acc'].append(percent_correct)
            lick_dict_probe['session_acc'].append(session_pc)
            lick_dict_probe['rewards'].append(np.sum(behav_no_probe['water']))
        else:
            pass
probe_df = pd.DataFrame(lick_dict_probe)

In [ ]:
## Plot probe performance for first and last day in A
avg_probe = probe_df.groupby(['day', 'group'], as_index=False).agg({'probe_acc': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=[''])

for group in ['Control', 'DREADD']:
    gdata = avg_probe[avg_probe['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers', marker_color=group_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=2.5)))
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(range=[0, 100])
fig.show()

### Look at probe accuracy across trials.

In [ ]:
bin_size = 1
lick_dict_trials = {'mouse': [], 'sex': [], 'group': [], 'session': [], 
                   'day': [], 'trial': [], 'lick_acc': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(natsorted(os.listdir(mpath))):
        behav = pd.read_feather(pjoin(mpath, session))
        if any(behav['probe']):
            behav_probe = behav[behav['probe']]
            reward_one, reward_two = behav_probe['reward_one'].unique()[0], behav_probe['reward_two'].unique()[0]
            trial_acc = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=True)

            if bin_size > 1:
                binned_acc = ctb.bin_data(trial_acc, bin_size)
            else:
                binned_acc = trial_acc
            
            for trial, val in enumerate(binned_acc):
                lick_dict_trials['mouse'].append(mouse)
                lick_dict_trials['sex'].append(sex)
                lick_dict_trials['group'].append(group)
                lick_dict_trials['day'].append(idx + 1)
                lick_dict_trials['session'].append(behav_probe['session_two'].unique()[0])
                lick_dict_trials['trial'].append(trial + 1)
                lick_dict_trials['lick_acc'].append(val)
probe_trials_df = pd.DataFrame(lick_dict_trials)

In [ ]:
## Plot probe performance across trials for days 1 and 5
avg_probe_trials = probe_trials_df.groupby(['group', 'day', 'trial'], as_index=False).agg({'lick_acc': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Trial', y_title='', rows=1, columns=2, width=800,
                               shared_x=True, shared_y=True, titles=['Day 1', 'Day 5'])
for group in ['Control', 'DREADD']:
    gdata = avg_probe_trials[avg_probe_trials['group'] == group]
    for idx, day in enumerate([1, 5]):
        day_data = gdata[gdata['day'] == day]
        upper = day_data['lick_acc']['mean'] + day_data['lick_acc']['sem']
        lower = day_data['lick_acc']['mean'] - day_data['lick_acc']['sem']
        fig.add_trace(go.Scattergl(x=day_data['trial'], y=day_data['lick_acc']['mean'], mode='lines', 
                                line_color=group_colors_dict[group], name=group, legendgroup=group, showlegend=False), row=1, col=idx + 1)
        fig.add_trace(go.Scatter(x=day_data['trial'], y=upper, mode='lines', marker=dict(color=error_dict[group]),
                                                name='Upper Bound', line=dict(width=0), showlegend=False), row=1, col=idx + 1)
        fig.add_trace(go.Scatter(x=day_data['trial'], y=lower, mode='lines', marker=dict(color=error_dict[group]),
                                name='Lower Bound', line=dict(width=0), showlegend=False, fillcolor=error_dict[group], fill='tonexty'), row=1, col=idx + 1)
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy (%)', col=1)
fig.update_yaxes(range=[0, 100])
fig.update_xaxes(dtick=1)
fig.show()

In [ ]:
gdata